In [1]:
import pygame
import random

# 定数
SCREEN_WIDTH = 800
SCREEN_HEIGHT = 600
PLAYER_COLOR = (0, 128, 255)  # 青
ENEMY_COLOR = (255, 0, 0)    # 赤
BULLET_COLOR = (255, 255, 0) # 黄色
BACKGROUND_COLOR = (0, 0, 0) # 黒
PLAYER_SIZE = 50
ENEMY_SIZE = 30
BULLET_SIZE = 10
PLAYER_SPEED = 5
ENEMY_SPEED = 2
BULLET_SPEED = 7
ENEMY_SPAWN_RATE = 60 # 60フレームごと (約1秒に1回)

# Pygameの初期化
pygame.init()

# 画面の設定
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("縦スクロールシューティング")

# 時計（フレームレート制御用）
clock = pygame.time.Clock()

# プレイヤー クラス
class Player(pygame.sprite.Sprite):
    def __init__(self):
        super().__init__()
        # プレイヤーの形状（四角形）
        self.image = pygame.Surface([PLAYER_SIZE, PLAYER_SIZE])
        self.image.fill(PLAYER_COLOR)
        self.rect = self.image.get_rect()
        # 初期位置は画面下部中央
        self.rect.x = (SCREEN_WIDTH - PLAYER_SIZE) // 2
        self.rect.y = SCREEN_HEIGHT - PLAYER_SIZE - 10
        self.speed_x = 0

    def update(self):
        # 左右移動
        self.rect.x += self.speed_x
        # 画面端での移動制限
        if self.rect.left < 0:
            self.rect.left = 0
        if self.rect.right > SCREEN_WIDTH:
            self.rect.right = SCREEN_WIDTH

    def shoot(self):
        # 弾を生成してスプライトグループに追加
        bullet = Bullet(self.rect.centerx, self.rect.top)
        all_sprites.add(bullet)
        bullets.add(bullet)

# 敵 クラス
class Enemy(pygame.sprite.Sprite):
    def __init__(self):
        super().__init__()
        # 敵の形状（四角形）
        self.image = pygame.Surface([ENEMY_SIZE, ENEMY_SIZE])
        self.image.fill(ENEMY_COLOR)
        self.rect = self.image.get_rect()
        # 初期位置は画面上部のランダムなX座標
        self.rect.x = random.randrange(SCREEN_WIDTH - ENEMY_SIZE)
        self.rect.y = random.randrange(-100, -ENEMY_SIZE) # 画面外から登場
        self.speed_y = ENEMY_SPEED

    def update(self):
        # 下に移動
        self.rect.y += self.speed_y
        # 画面外に出たら消滅
        if self.rect.top > SCREEN_HEIGHT + 10:
            self.kill() # スプライトグループから削除

# 弾 クラス
class Bullet(pygame.sprite.Sprite):
    def __init__(self, x, y):
        super().__init__()
        # 弾の形状（四角形）
        self.image = pygame.Surface([BULLET_SIZE, BULLET_SIZE])
        self.image.fill(BULLET_COLOR)
        self.rect = self.image.get_rect()
        # 発射位置
        self.rect.centerx = x
        self.rect.bottom = y
        self.speed_y = -BULLET_SPEED # 上に移動

    def update(self):
        # 上に移動
        self.rect.y += self.speed_y
        # 画面外に出たら消滅
        if self.rect.bottom < 0:
            self.kill() # スプライトグループから削除

# スプライトグループの作成
all_sprites = pygame.sprite.Group()
enemies = pygame.sprite.Group()
bullets = pygame.sprite.Group()

# プレイヤーの作成とスプライトグループへの追加
player = Player()
all_sprites.add(player)

# ゲームループ
running = True
enemy_spawn_timer = 0
score = 0 # スコア（今回は表示しませんが、将来的に拡張可能です）

while running:
    # フレームレート制御 (60 FPS)
    clock.tick(60)

    # イベント処理
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_LEFT:
                player.speed_x = -PLAYER_SPEED
            if event.key == pygame.K_RIGHT:
                player.speed_x = PLAYER_SPEED
            if event.key == pygame.K_SPACE:
                player.shoot()
        if event.type == pygame.KEYUP:
            if event.key == pygame.K_LEFT and player.speed_x < 0:
                player.speed_x = 0
            if event.key == pygame.K_RIGHT and player.speed_x > 0:
                player.speed_x = 0

    # 敵の出現
    enemy_spawn_timer += 1
    if enemy_spawn_timer >= ENEMY_SPAWN_RATE:
        enemy_spawn_timer = 0
        enemy = Enemy()
        all_sprites.add(enemy)
        enemies.add(enemy)

    # ゲームロジックの更新
    all_sprites.update()

    # 当たり判定
    # 弾と敵の当たり判定
    hits = pygame.sprite.groupcollide(enemies, bullets, True, True) # (敵グループ, 弾グループ, 敵を消すか, 弾を消すか)
    for hit in hits:
        score += 10 # スコア加算（今回は表示なし）
        # ここで爆発エフェクトなどを追加できます

    # プレイヤーと敵の当たり判定
    hits = pygame.sprite.spritecollide(player, enemies, False) # (プレイヤー, 敵グループ, 敵を消すか)
    if hits:
        running = False # ゲームオーバー

    # 描画処理
    screen.fill(BACKGROUND_COLOR) # 背景を黒で塗りつぶし
    all_sprites.draw(screen)      # 全てのスプライトを描画

    # 画面の更新
    pygame.display.flip()

# Pygameの終了処理
pygame.quit()

pygame 2.6.1 (SDL 2.28.4, Python 3.8.2)
Hello from the pygame community. https://www.pygame.org/contribute.html
